# 01 · Speech-to-text — faster-whisper (GPU)

**Kernel:** `JARVIS - whisper` (venv `C:\jarvis-venvs\whisper`, Python 3.11)

1. Live mic → webrtcvad endpointing (stops after 600 ms of silence) → `large-v3-turbo` on CUDA, `int8_float16`. Prints per-stage times and peak VRAM.
2. Benchmarks the same clip on GPU, then the CPU fallback (`int8`, 8 threads) for `large-v3-turbo` and `small`. GPU stays the default.

For clean VRAM numbers: close the Android emulators, then **Restart kernel → Run All**. Re-running the monitor cell after a model is loaded gives a wrong baseline.

In [ ]:
import os, sys, time, json, importlib.util, statistics
from pathlib import Path

os.environ.setdefault("HF_HOME", r"C:\jarvis-models\hf")   # model cache lives outside the project

OUT = Path.cwd() / "outputs"
OUT.mkdir(exist_ok=True)

MODEL = "large-v3-turbo"        # multilingual on purpose: never an .en model (English/Bengali/Hindi code-switching)
DEVICE, COMPUTE_TYPE = "cuda", "int8_float16"
BEAM_SIZE = 5
LANGUAGE = None                 # None = auto-detect
SILENCE_MS = 600                # endpoint after this much trailing silence
VAD_AGGRESSIVENESS = 2          # webrtcvad, 0 (lenient) .. 3 (strict)
MIC_DEVICE = None               # sounddevice input index or name; None = Windows default

assert sys.version_info[:2] in [(3, 11), (3, 12)], sys.version
print(sys.executable)

In [ ]:
# --- VRAM monitor: create BEFORE anything touches CUDA ---------------------------------
# process MB : this process's dedicated memory on the NVIDIA adapter, from the Windows
#              "GPU Process Memory" counter (what Task Manager shows). NVML cannot see
#              per-process usage under WDDM; this counter can. Headline figure.
# device delta MB : NVML total used minus the baseline. Includes other apps - cross-check.
import ctypes, threading
from ctypes import wintypes
import pynvml

class _FmtValue(ctypes.Structure):
    _fields_ = [("CStatus", wintypes.DWORD), ("largeValue", ctypes.c_longlong)]

class _FmtItem(ctypes.Structure):
    _fields_ = [("szName", wintypes.LPWSTR), ("FmtValue", _FmtValue)]

class PdhWildcard:
    """Every instance of a wildcard Windows performance counter, as {instance: bytes}."""
    def __init__(self, path):
        self.pdh, self.path = ctypes.WinDLL("pdh"), path
        self.query, self.counter = wintypes.HANDLE(), wintypes.HANDLE()
        self.reopen()

    def reopen(self):
        if self.query.value:
            self.pdh.PdhCloseQuery(self.query)
        self.query, self.counter = wintypes.HANDLE(), wintypes.HANDLE()
        if self.pdh.PdhOpenQueryW(None, None, ctypes.byref(self.query)) != 0:
            raise OSError("PdhOpenQueryW failed")
        if self.pdh.PdhAddEnglishCounterW(self.query, self.path, None, ctypes.byref(self.counter)) != 0:
            raise OSError(f"cannot open counter {self.path}")

    def read(self):
        if self.pdh.PdhCollectQueryData(self.query) != 0:
            return {}
        size, count = wintypes.DWORD(0), wintypes.DWORD(0)
        self.pdh.PdhGetFormattedCounterArrayW(self.counter, 0x400, ctypes.byref(size), ctypes.byref(count), None)
        if size.value == 0:
            return {}
        buf = (ctypes.c_byte * size.value)()
        if self.pdh.PdhGetFormattedCounterArrayW(self.counter, 0x400, ctypes.byref(size), ctypes.byref(count), buf) != 0:
            return {}
        items = ctypes.cast(buf, ctypes.POINTER(_FmtItem * count.value)).contents
        return {i.szName: i.FmtValue.largeValue for i in items if i.szName}

class VramMonitor:
    def __init__(self, interval=0.05):
        pynvml.nvmlInit()
        self.handle = pynvml.nvmlDeviceGetHandleByIndex(0)
        self.baseline = pynvml.nvmlDeviceGetMemoryInfo(self.handle).used
        # Hybrid laptop: pick the adapter LUID whose usage matches NVML, i.e. the RTX 4060.
        adapters = PdhWildcard(r"\GPU Adapter Memory(*)\Dedicated Usage").read()
        nvidia = min(adapters, key=lambda name: abs(adapters[name] - self.baseline))
        self.luid = nvidia.split("_phys")[0]
        self.prefix = f"pid_{os.getpid()}_{self.luid}"
        self.procs = PdhWildcard(r"\GPU Process Memory(*)\Dedicated Usage")
        self.peak_proc = self.peak_dev = 0
        self.stages, self._last_reopen = [], time.perf_counter()
        self._lock, self._stop = threading.Lock(), threading.Event()
        threading.Thread(target=self._run, args=(interval,), daemon=True).start()
        print(f"{pynvml.nvmlDeviceGetName(self.handle)} | adapter {self.luid} | "
              f"baseline used by other apps: {self.baseline / 2**20:.0f} MB")

    def _run(self, interval):
        while not self._stop.is_set():
            self.sample()
            time.sleep(interval)

    def sample(self):
        mine = [v for n, v in self.procs.read().items() if n.startswith(self.prefix)]
        if not mine and time.perf_counter() - self._last_reopen > 1:
            self.procs.reopen()  # our instance only appears once CUDA creates a context
            self._last_reopen = time.perf_counter()
        proc = max(mine, default=0)
        dev = pynvml.nvmlDeviceGetMemoryInfo(self.handle).used - self.baseline
        with self._lock:
            self.peak_proc, self.peak_dev = max(self.peak_proc, proc), max(self.peak_dev, dev)
        return proc, dev

    def stage(self, label):
        proc, dev = self.sample()
        entry = {"stage": label, "process_now_mb": round(proc / 2**20), "process_peak_mb": round(self.peak_proc / 2**20),
                 "device_delta_now_mb": round(dev / 2**20), "device_delta_peak_mb": round(self.peak_dev / 2**20)}
        self.stages.append(entry)
        print(f"VRAM [{label}] now {entry['process_now_mb']} MB, peak {entry['process_peak_mb']} MB "
              f"(device delta now {entry['device_delta_now_mb']}, peak {entry['device_delta_peak_mb']})")
        return entry

    def summary(self):
        self.sample()
        return {"peak_process_mb": round(self.peak_proc / 2**20), "peak_device_delta_mb": round(self.peak_dev / 2**20),
                "baseline_other_apps_mb": round(self.baseline / 2**20), "stages": self.stages}

vram = VramMonitor()

In [ ]:
# CTranslate2 needs cuBLAS 12 + cuDNN 9. cuBLAS comes from the nvidia-cublas-cu12 wheel; the cuDNN 9 DLLs were
# copied into <venv>\cudnn9 from the local torch cu128 wheel (see SETUP.md). The nvidia-cudnn wheel is used if present.
dll_dirs = [Path(sys.prefix) / "cudnn9"]
for pkg in ("nvidia.cublas", "nvidia.cudnn"):
    spec = importlib.util.find_spec(pkg)
    if spec and spec.submodule_search_locations:
        dll_dirs.append(Path(list(spec.submodule_search_locations)[0]) / "bin")
for d in [d for d in dll_dirs if d.is_dir()]:
    os.add_dll_directory(str(d))
    os.environ["PATH"] = str(d) + os.pathsep + os.environ["PATH"]
    print("CUDA DLLs from", d)

t = time.perf_counter()
import numpy as np, sounddevice as sd, soundfile as sf, webrtcvad
import ctranslate2, faster_whisper
from faster_whisper import WhisperModel, decode_audio
import_ms = (time.perf_counter() - t) * 1000
print(f"faster-whisper {faster_whisper.__version__} | ctranslate2 {ctranslate2.__version__} | "
      f"CUDA devices seen: {ctranslate2.get_cuda_device_count()} | imports {import_ms:.0f} ms")

In [ ]:
t = time.perf_counter()
model = WhisperModel(MODEL, device=DEVICE, compute_type=COMPUTE_TYPE)
load_ms = (time.perf_counter() - t) * 1000
print(f"load {MODEL} [{DEVICE}/{COMPUTE_TYPE}]: {load_ms:.0f} ms  (the very first run includes a ~1.6 GB download)")
vram.stage("after_load")

# First inference compiles/initialises CUDA kernels; do it now so the live test measures steady state.
t = time.perf_counter()
list(model.transcribe(np.zeros(16000, dtype=np.float32), language="en", beam_size=BEAM_SIZE)[0])
print(f"CUDA warm-up: {(time.perf_counter() - t) * 1000:.0f} ms")
vram.stage("after_warmup")

## Microphone + VAD endpointing
Check the input device first — Windows default should be the *Intel Smart Sound* microphone array.

In [ ]:
print(sd.query_devices(kind="input"))

def record_utterance(silence_ms=SILENCE_MS, wait_s=15, max_s=30):
    """Waits for speech, records until `silence_ms` of silence. Returns (float32 audio @16 kHz, timings)."""
    import collections, queue
    rate, frame_ms = 16000, 30
    frame_len = rate * frame_ms // 1000
    vad, frames = webrtcvad.Vad(VAD_AGGRESSIVENESS), queue.Queue()
    pre_roll = collections.deque(maxlen=10)      # keep 300 ms before the trigger
    window = collections.deque(maxlen=10)
    voiced, silent_run, t_speech, timings = [], 0, None, {}

    t_open = time.perf_counter()
    with sd.RawInputStream(samplerate=rate, blocksize=frame_len, channels=1, dtype="int16",
                           device=MIC_DEVICE, callback=lambda data, *_: frames.put(bytes(data))):
        t_listen = time.perf_counter()
        timings["mic_open_ms"] = round((t_listen - t_open) * 1000)
        print("Listening - speak now...")
        while True:
            try:
                frame = frames.get(timeout=0.5)
            except queue.Empty:
                frame = None
            now = time.perf_counter()
            if t_speech is None and now - t_listen > wait_s:
                return None, timings
            if frame is None or len(frame) != frame_len * 2:
                continue
            speech = vad.is_speech(frame, rate)
            if t_speech is None:
                pre_roll.append(frame); window.append(speech)
                if sum(window) >= 6:                 # ~180 ms of speech in the last 300 ms
                    t_speech = now
                    voiced.extend(pre_roll)
                continue
            voiced.append(frame)
            silent_run = 0 if speech else silent_run + 1
            if silent_run * frame_ms >= silence_ms or now - t_speech > max_s:
                timings.update(wait_for_speech_ms=round((t_speech - t_listen) * 1000),
                               speech_start_to_endpoint_ms=round((now - t_speech) * 1000),
                               trailing_silence_ms=silent_run * frame_ms)
                break
    audio = np.frombuffer(b"".join(voiced), dtype=np.int16).astype(np.float32) / 32768
    return audio, timings

def transcribe(m, audio, multilingual=False):
    t0 = time.perf_counter()
    segments, info = m.transcribe(audio, language=LANGUAGE, beam_size=BEAM_SIZE, vad_filter=False,
                                  condition_on_previous_text=False, multilingual=multilingual)
    first, texts = None, []
    for seg in segments:                             # segments is lazy: decoding happens here
        first = first or time.perf_counter() - t0
        texts.append(seg.text.strip())
    total = time.perf_counter() - t0
    return {"text": " ".join(texts), "language": info.language, "language_p": round(info.language_probability, 2),
            "first_segment_ms": round((first or total) * 1000), "total_ms": round(total * 1000),
            "audio_s": round(len(audio) / 16000, 2), "rtf": round(total / max(len(audio) / 16000, 1e-6), 3)}

## Live test — speak a sentence
Try one pure-English and one code-switched sentence (English + Bengali/Hindi). Set `multilingual=True` below
to let Whisper re-detect language per segment instead of once for the whole clip.

In [ ]:
audio, rec = record_utterance()
if audio is None:
    print("No speech detected - check MIC_DEVICE and run this cell again.")
else:
    sf.write(OUT / "mic_last.wav", audio, 16000)          # reused by the benchmark below
    stt = transcribe(model, audio, multilingual=False)
    live = {**rec, "transcribe_first_segment_ms": stt["first_segment_ms"], "transcribe_total_ms": stt["total_ms"]}
    print(f"\n>>> {stt['text']}\n    [{stt['language']} p={stt['language_p']}, {stt['audio_s']} s of audio]\n")
    for k, v in live.items():
        print(f"  {k:32s} {v:>6} ms")
    print(f"  {'endpoint -> text (user-perceived)':32s} {stt['total_ms']:>6} ms  (+ {SILENCE_MS} ms silence wait)")
    vram.stage("after_live_transcribe")

## Benchmark: GPU vs CPU fallback on the same clip
Uses your last mic recording if there is one, otherwise 15 s of the reference audio.
One cold run, then 3 timed warm runs; the median is reported. CPU models use no VRAM.

In [ ]:
REFERENCE = Path(r"P:\Coding\App Development\MyProjects\JARVIS\JARVIS voice clone.mp3")
if (OUT / "mic_last.wav").exists():
    bench_source, bench_audio = "mic_last.wav", decode_audio(str(OUT / "mic_last.wav"), 16000)
else:
    bench_source, bench_audio = "reference mp3 30-45 s", decode_audio(str(REFERENCE), 16000)[30 * 16000:45 * 16000]
print(f"benchmark clip: {bench_source}, {len(bench_audio) / 16000:.1f} s")

def bench(m, runs=3):
    cold = transcribe(m, bench_audio)
    warm = [transcribe(m, bench_audio) for _ in range(runs)]
    return {"cold_ms": cold["total_ms"], "warm_median_ms": round(statistics.median(r["total_ms"] for r in warm)),
            "warm_first_segment_ms": round(statistics.median(r["first_segment_ms"] for r in warm)),
            "rtf": round(statistics.median(r["rtf"] for r in warm), 3), "text": warm[-1]["text"]}

bench_results = {f"gpu {MODEL} {COMPUTE_TYPE}": {"load_ms": round(load_ms), **bench(model)}}
vram.stage("after_gpu_bench")

for name in ["large-v3-turbo", "small"]:
    t = time.perf_counter()
    cpu_model = WhisperModel(name, device="cpu", compute_type="int8", cpu_threads=8)
    cpu_load = round((time.perf_counter() - t) * 1000)
    bench_results[f"cpu {name} int8 x8"] = {"load_ms": cpu_load, **bench(cpu_model)}
    del cpu_model
    print(f"done: cpu {name}")

print(f"\n{'config':34s} {'load':>7} {'cold':>7} {'warm':>7} {'1st seg':>8} {'RTF':>6}")
for k, r in bench_results.items():
    print(f"{k:34s} {r['load_ms']:>7} {r['cold_ms']:>7} {r['warm_median_ms']:>7} {r['warm_first_segment_ms']:>8} {r['rtf']:>6}")

In [ ]:
summary = {"model": MODEL, "compute_type": COMPUTE_TYPE, "beam_size": BEAM_SIZE,
           "faster_whisper": faster_whisper.__version__, "ctranslate2": ctranslate2.__version__,
           "live": globals().get("live"), "bench_source": bench_source, "bench": bench_results, "vram": vram.summary()}
(OUT / "whisper_results.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Peak VRAM (Whisper, GPU): {summary['vram']['peak_process_mb']} MB process "
      f"/ {summary['vram']['peak_device_delta_mb']} MB device delta")